In [1]:
!pip install langid -q
!pip uninstall -y transformers accelerate datasets
!pip install -q transformers==4.44.2 accelerate==0.34.2 datasets==2.21.0 scikit-learn scipy

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: datasets 4.8.3
Uninstalling datasets-4.8.3:
  Successfully uninstalled datasets-4.8.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.7 MB/

In [2]:
import pandas as pd
import numpy as np
import re
import os
import subprocess
import sys

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier, SGDClassifier

In [3]:
import torch
import itertools
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from scipy.special import softmax


# os.environ["WANDB_DISABLED"] = "true"

2026-05-18 16:22:13.346519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779121333.518825     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779121333.573936     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779121333.975929     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779121333.975970     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779121333.975973     114 computation_placer.cc:177] computation placer alr

In [4]:
DATA_PATH = "/kaggle/input/datasets/arailymakhmet/merged/merged_dataset.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.head())
print(df["label"].value_counts())
print(df["lang_label"].value_counts())

df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

(81648, 3)
                                                text  label lang_label
0  году специальную экономическую зону морпорт ак...      1         ru
1                      мынау жақсы екен орнатыңыздар      1         kk
2  посмотрела сорок восемь серий двадцать восьмой...      1         ru
3                                     гадость гадкая      0         ru
4  обычно всё устраивает это пельмени ломаные вар...      0         ru
label
1    40824
0    40824
Name: count, dtype: int64
lang_label
ru    40824
kk    40824
Name: count, dtype: int64


In [5]:

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (65295,)
Test: (16324,)


In [6]:
results = []

def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1_macro = f1_score(y_test, preds, average="macro")
    f1_weighted = f1_score(y_test, preds, average="weighted")

    results.append({
        "model": name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

    print("=" * 70)
    print(name)
    print("=" * 70)
    print("Accuracy:", acc)
    print("F1 macro:", f1_macro)
    print("F1 weighted:", f1_weighted)
    print("\nClassification report:")
    print(classification_report(y_test, preds))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, preds))

In [7]:
#TF-IDF + RidgeClassifier
ridge_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=150000,
        sublinear_tf=True
    )),
    ("clf", RidgeClassifier(
        alpha=1.0,
        class_weight="balanced"
    ))
])

ridge_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + RidgeClassifier",
    ridge_model,
    X_test,
    y_test
)

Word TF-IDF + RidgeClassifier
Accuracy: 0.8431144327370742
F1 macro: 0.843111295233526
F1 weighted: 0.8431113811925273

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      8163
           1       0.85      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6918 1245]
 [1316 6845]]


In [8]:
#TF-IDF + SGDClassifier
sgd_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=150000,
        sublinear_tf=True
    )),
    ("clf", SGDClassifier(
        loss="hinge",          # linear SVM style
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        class_weight="balanced",
        random_state=42
    ))
])

sgd_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + SGDClassifier",
    sgd_model,
    X_test,
    y_test
)

Word TF-IDF + SGDClassifier
Accuracy: 0.842930654251409
F1 macro: 0.8429302557901891
F1 weighted: 0.842930225139326

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.84      0.84      8163
           1       0.84      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6867 1296]
 [1268 6893]]


In [9]:
#Word TF-IDF + Char TF-IDF
word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=100000,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=150000,
    sublinear_tf=True
)

combined_features = FeatureUnion([
    ("word_tfidf", word_tfidf),
    ("char_tfidf", char_tfidf)
])

word_char_ridge_model = Pipeline([
    ("features", combined_features),
    ("clf", RidgeClassifier(
        alpha=1.0,
        class_weight="balanced"
    ))
])

word_char_ridge_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + Char TF-IDF + RidgeClassifier",
    word_char_ridge_model,
    X_test,
    y_test
)

Word TF-IDF + Char TF-IDF + RidgeClassifier
Accuracy: 0.8472188189169321
F1 macro: 0.847218231808502
F1 weighted: 0.8472182685027789

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      8163
           1       0.85      0.85      0.85      8161

    accuracy                           0.85     16324
   macro avg       0.85      0.85      0.85     16324
weighted avg       0.85      0.85      0.85     16324

Confusion matrix:
[[6931 1232]
 [1262 6899]]


In [10]:
word_char_sgd_model = Pipeline([
    ("features", combined_features),
    ("clf", SGDClassifier(
        loss="hinge",
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        class_weight="balanced",
        random_state=42
    ))
])

word_char_sgd_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + Char TF-IDF + SGDClassifier",
    word_char_sgd_model,
    X_test,
    y_test
)

Word TF-IDF + Char TF-IDF + SGDClassifier
Accuracy: 0.8459323695172751
F1 macro: 0.8459271513241204
F1 weighted: 0.8459270414674224

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      8163
           1       0.84      0.85      0.85      8161

    accuracy                           0.85     16324
   macro avg       0.85      0.85      0.85     16324
weighted avg       0.85      0.85      0.85     16324

Confusion matrix:
[[6857 1306]
 [1209 6952]]


In [11]:
!pip install fasttext -q

In [12]:
import fasttext

In [13]:
fasttext_train_path = "/kaggle/working/fasttext_train.txt"
fasttext_test_path = "/kaggle/working/fasttext_test.txt"

def clean_fasttext_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_fasttext = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_fasttext = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

with open(fasttext_train_path, "w", encoding="utf-8") as f:
    for _, row in train_fasttext.iterrows():
        label = int(row["label"])
        text = clean_fasttext_text(row["text"])
        f.write(f"__label__{label} {text}\n")

with open(fasttext_test_path, "w", encoding="utf-8") as f:
    for _, row in test_fasttext.iterrows():
        label = int(row["label"])
        text = clean_fasttext_text(row["text"])
        f.write(f"__label__{label} {text}\n")

In [14]:
!pip install fasttext -q --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but 

In [15]:
# Train fastText

fasttext_model = fasttext.train_supervised(
    input=fasttext_train_path,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    minn=3,
    maxn=5,
    dim=100,
    loss="softmax",
    verbose=2
)

# Evaluate fastText

def predict_fasttext(model, texts):
    preds = []
    for text in texts:
        text = clean_fasttext_text(text)
        pred = model.predict(text)[0][0]
        pred = int(pred.replace("__label__", ""))
        preds.append(pred)
    return np.array(preds)

fasttext_preds = predict_fasttext(fasttext_model, X_test)

acc = accuracy_score(y_test, fasttext_preds)
f1_macro = f1_score(y_test, fasttext_preds, average="macro")
f1_weighted = f1_score(y_test, fasttext_preds, average="weighted")

results.append({
    "model": "fastText supervised",
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
})

print("=" * 70)
print("fastText supervised")
print("=" * 70)
print("Accuracy:", acc)
print("F1 macro:", f1_macro)
print("F1 weighted:", f1_weighted)
print("\nClassification report:")
print(classification_report(y_test, fasttext_preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, fasttext_preds))

Read 1M words
Number of words:  209350
Number of labels: 2
Progress: 100.0% words/sec/thread:  130635 lr:  0.000000 avg.loss:  0.187485 ETA:   0h 0m 0s  3.4% words/sec/thread:  100069 lr:  0.482805 avg.loss:  0.466527 ETA:   0h 2m 1s 18.5% words/sec/thread:  124178 lr:  0.407337 avg.loss:  0.381551 ETA:   0h 1m22s 0.360099 ETA:   0h 1m17s 31.5% words/sec/thread:  127386 lr:  0.342464 avg.loss:  0.321383 ETA:   0h 1m 7s 1m 0s


ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [16]:
results_df = pd.DataFrame(results).sort_values(
    by="f1_macro",
    ascending=False
)

results_df

,model,accuracy,f1_macro,f1_weighted
2,Word TF-IDF + Char TF-IDF + RidgeClassifier,0.847219,0.847218,0.847218
3,Word TF-IDF + Char TF-IDF + SGDClassifier,0.845932,0.845927,0.845927
0,Word TF-IDF + RidgeClassifier,0.843114,0.843111,0.843111
1,Word TF-IDF + SGDClassifier,0.842931,0.842930,0.842930


In [18]:
import joblib

joblib.dump(ridge_model, "/kaggle/working/ridge_tfidf_model.pkl")
joblib.dump(sgd_model, "/kaggle/working/sgd_tfidf_model.pkl")
joblib.dump(word_char_ridge_model, "/kaggle/working/word_char_ridge_model.pkl")
joblib.dump(word_char_sgd_model, "/kaggle/working/word_char_sgd_model.pkl")


results_df.to_csv("/kaggle/working/model_results.csv", index=False)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

results = []

def add_metrics(model_name, y_true, y_pred):
    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    })

def show_report(model_name, y_true, y_pred):
    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

ridge_preds = ridge_model.predict(X_test)
add_metrics("TF-IDF + RidgeClassifier", y_test, ridge_preds)
show_report("TF-IDF + RidgeClassifier", y_test, ridge_preds)

sgd_preds = sgd_model.predict(X_test)
add_metrics("TF-IDF + SGDClassifier", y_test, sgd_preds)
show_report("TF-IDF + SGDClassifier", y_test, sgd_preds)

word_char_ridge_preds = word_char_ridge_model.predict(X_test)
add_metrics("Word + Char TF-IDF + RidgeClassifier", y_test, word_char_ridge_preds)
show_report("Word + Char TF-IDF + RidgeClassifier", y_test, word_char_ridge_preds)

word_char_sgd_preds = word_char_sgd_model.predict(X_test)
add_metrics("Word + Char TF-IDF + SGDClassifier", y_test, word_char_sgd_preds)
show_report("Word + Char TF-IDF + SGDClassifier", y_test, word_char_sgd_preds)

TF-IDF + RidgeClassifier
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      8163
           1       0.85      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6918 1245]
 [1316 6845]]
TF-IDF + SGDClassifier
              precision    recall  f1-score   support

           0       0.84      0.84      0.84      8163
           1       0.84      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6867 1296]
 [1268 6893]]
Word + Char TF-IDF + RidgeClassifier
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      8163
           1       0.85      0.85      0.85      8161

    acc

In [31]:
# 1. Metric computation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probabilities = softmax(logits, axis=1)[:, 1] # Probabilities for the Positive class

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "roc_auc": roc_auc_score(labels, probabilities),
        "precision": precision_score(labels, predictions, average="binary", zero_division=0),
        "recall": recall_score(labels, predictions, average="binary", zero_division=0)
    }

# 2. Fast training function (NO grid search)
def train_transformer(model_name, df, output_dir, batch_size, lr=2e-5, num_epochs=3, max_length=128):
    print(f"\n{'='*50}")
    print(f"FAST TRAINING: {model_name}")
    print(f"{'='*50}")
    
    # Train/validation split (85/15)
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])

    # Convert to Hugging Face Dataset format
    train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
    val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

    train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["__index_level_0__"] if "__index_level_0__" in train_dataset.column_names else None)
    val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["__index_level_0__"] if "__index_level_0__" in val_dataset.column_names else None)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        weight_decay=0.01,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=num_epochs,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        fp16=torch.cuda.is_available(), # Mixed precision to speed up P100!
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Train
    trainer.train()
    
    # Save BEST model
    final_path = f"{output_dir}/best_model"
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)
    
    # Print final metrics
    eval_metrics = trainer.evaluate()
    print(f"\n✅ Final metrics for {model_name}:\n{eval_metrics}\n")

    return trainer


In [32]:
df = df[["text", "label"]].dropna().copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)
df = df.reset_index(drop=True)

In [33]:
mbert_trainer = train_transformer(
    model_name="bert-base-multilingual-cased",
    df=df,
    output_dir="./models/mbert_all_languages",
    batch_size=32,
    lr=2e-5,
    num_epochs=3,
    max_length=128
)


FAST TRAINING: bert-base-multilingual-cased


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/69376 [00:00<?, ? examples/s]

Map:   0%|          | 0/12243 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Roc Auc,Precision,Recall
1,0.380600,0.365020,0.846034,0.845979,0.922700,0.859593,0.827152
2,0.312500,0.366235,0.849710,0.849639,0.926496,0.835029,0.871590
3,0.259600,0.374705,0.853141,0.853136,0.927564,0.849023,0.859010


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


✅ Final metrics for bert-base-multilingual-cased:
{'eval_loss': 0.3747050166130066, 'eval_accuracy': 0.8531405701217022, 'eval_f1_macro': 0.8531356309122597, 'eval_roc_auc': 0.9275637595115085, 'eval_precision': 0.8490230905861457, 'eval_recall': 0.8590099656918804, 'eval_runtime': 62.0461, 'eval_samples_per_second': 197.321, 'eval_steps_per_second': 1.547, 'epoch': 3.0}



In [36]:
DATA_PATH = "/kaggle/input/datasets/arailymakhmet/merged/merged_dataset.csv"

df_full = pd.read_csv(DATA_PATH)

df_full = df_full.dropna(subset=["text", "label", "lang_label"]).copy()
df_full["text"] = df_full["text"].astype(str)
df_full["label"] = df_full["label"].astype(int)
df_full["lang_label"] = df_full["lang_label"].astype(str)

print(df_full.columns)
print(df_full["lang_label"].value_counts())

Index(['text', 'label', 'lang_label'], dtype='object')
lang_label
kk    40824
ru    40795
Name: count, dtype: int64


In [37]:
df_kk = df_full[df_full["lang_label"] == "kk"].copy()
df_ru = df_full[df_full["lang_label"] == "ru"].copy()

print("KK:", df_kk.shape)
print("RU:", df_ru.shape)

KK: (40824, 3)
RU: (40795, 3)


In [38]:
mbert_kk_trainer = train_transformer(
    model_name="bert-base-multilingual-cased",
    df=df_kk,
    output_dir="./models/mbert_kk",
    batch_size=32,
    lr=2e-5,
    num_epochs=3
)

rubert_ru_trainer = train_transformer(
    model_name="cointegrated/rubert-tiny2",
    df=df_ru,
    output_dir="./models/rubert_ru",
    batch_size=64,
    lr=5e-5,
    num_epochs=3
)


FAST TRAINING: bert-base-multilingual-cased


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/34700 [00:00<?, ? examples/s]

Map:   0%|          | 0/6124 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Roc Auc,Precision,Recall
1,No log,0.309531,0.864216,0.864173,0.942049,0.877371,0.846682
2,0.324100,0.309089,0.869608,0.869204,0.949003,0.915472,0.814318
3,0.324100,0.294648,0.878758,0.878741,0.950786,0.887588,0.867277


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


✅ Final metrics for cointegrated/rubert-tiny2:
{'eval_loss': 0.29464849829673767, 'eval_accuracy': 0.8787581699346405, 'eval_f1_macro': 0.8787413867355728, 'eval_roc_auc': 0.9507859638158361, 'eval_precision': 0.8875878220140515, 'eval_recall': 0.8672768878718535, 'eval_runtime': 2.648, 'eval_samples_per_second': 2311.162, 'eval_steps_per_second': 9.063, 'epoch': 3.0}

